# Luna et al. 2017 grid and seven ratios

Reference: Luna, Aranda & Santos (2017), International Journal of Osteoarchaeology 27:898–911, [doi:10.1002/oa.2604](https://onlinelibrary.wiley.com/doi/abs/10.1002/oa.2604), pp.900–902, Figure 2.

Original study used left ilia. Here right-only images will be horizontally reflected automatically and recorded as a right-derived methodological extension, not an observed left bone. No bilateral averaging. If a left exists but is unusable, explicitly override to R and record the reason.

Automatic landmarks are PROPOSALS, not validated anatomy. Masks omit anatomical context; confirm against original photographs where needed. In particular gamma is the superior point of the ANTERIOR area, not necessarily the global silhouette extreme. Beta is the auricular edge angle, not necessarily the deepest convexity defect.

Requires masks already approximately vertically oriented. Editor offers residual rotation; no image stretching. Ratios are dimensionless; segment lengths are in working-image pixels. No sex classifier included. Sample <7 or >18 years is outside the original study age range. Predictions would not be validation without independent sex labels.

Unsaved editor drafts are in memory only; save before restarting or rerunning the editor. No automatic loading of previous approved annotations is added, to avoid unintentionally unblinding repeat observations.


## Installation (once)
Uncomment the installation command if needed, run it, then restart the kernel. Run the configuration cell next to enable the interactive backend.

In [ ]:
# %pip install -U numpy pandas Pillow scipy opencv-python matplotlib ipywidgets ipympl scikit-learn

In [ ]:
from pathlib import Path
import re, json, hashlib, uuid
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage
import cv2
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

import ipympl
%matplotlib ipympl

import matplotlib
import sys

print("Python:", sys.executable)
print("ipympl:", ipympl.__version__)
print("Backend:", matplotlib.get_backend())

ROOT = Path.cwd()
PROJECT_ROOT = ROOT / "morphometrixnonadult" if (ROOT / "morphometrixnonadult").exists() else ROOT
MASK_DIR = PROJECT_ROOT / "masks"
PHOTO_DIR = PROJECT_ROOT / "f"
OUTPUT = PROJECT_ROOT / "luna_output"
OBSERVER_ID = "observer_1"
SESSION_ID = "session_1"  # change for repeat observations
RIGHT_ALREADY_REFLECTED = False
BACKGROUND_TOLERANCE = 20
SIDE_OVERRIDES = {}  # e.g. {'8': 'R'} only if left is unsuitable
SIDE_OVERRIDE_REASONS = {}  # e.g. {'8': 'left border damaged'}
for sub in ["draft_overlays", "reviewed_overlays", "annotations"]:
    (OUTPUT / sub).mkdir(parents=True, exist_ok=True)

metadata_path = ROOT / "consolidated_metadata.csv"
if not metadata_path.exists():
    metadata_path = PROJECT_ROOT / "consolidated_metadata.csv"
metadata = pd.read_csv(
    metadata_path,
    sep=None,
    engine="python",
    dtype=str,
    keep_default_na=False,
)
metadata["individual_id"] = metadata.individual_id.str.strip().str.replace(",", ".", regex=False)
assert metadata.individual_id.is_unique

pattern = re.compile(
    r"^(?:f)?(?P<id>\d+(?:[.,-]\d+)?)[_ ]?(?P<side>[LR])(?:[^A-Za-z].*)?$",
    re.I,
)
photo_pattern = re.compile(r"^f(?P<id>\d+(?:-\d+)?)P\d+$", re.I)
valid_exts = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".JPG", ".JPEG", ".PNG", ".TIF", ".TIFF", ".BMP"}

mask_rows = []
for p in sorted(MASK_DIR.rglob("*")):
    if p.suffix.lower() not in {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}:
        continue
    m = pattern.match(p.stem)
    if m:
        mask_rows.append(
            {
                "individual_id": m["id"].replace("-", ".").replace(",", "."),
                "side": m["side"].upper(),
                "path": str(p),
                "filename": p.name,
                "stem": p.stem,
            }
        )

files = pd.DataFrame(mask_rows)
assert len(files), f"No masks found in {MASK_DIR}"
assert not files.duplicated(["individual_id", "side"]).any(), "Resolve duplicate individual-side mask files"

assert PHOTO_DIR.exists(), f"Photo directory not found: {PHOTO_DIR}"
photo_rows = []
for p in sorted(PHOTO_DIR.rglob("*")):
    if p.suffix.lower() not in {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}:
        continue
    m = photo_pattern.match(p.stem)
    if m:
        photo_rows.append(
            {
                "individual_id": m["id"].replace("-", "."),
                "reference_path": str(p),
                "reference_filename": p.name,
                "reference_stem": p.stem,
            }
        )
photos = pd.DataFrame(photo_rows)
assert len(photos), f"No reference photos found in {PHOTO_DIR}"
assert not photos.duplicated(["individual_id", "reference_filename"]).any(), "Resolve duplicate photo filenames"

selected = []
metadata_ids = set(metadata.individual_id)
for ident, g in files.groupby("individual_id"):
    if ident not in metadata_ids:
        continue
    side = SIDE_OVERRIDES.get(ident, "L" if "L" in set(g.side) else "R")
    assert side in set(g.side), f"Override side unavailable for {ident}"
    if ident in SIDE_OVERRIDES:
        assert SIDE_OVERRIDE_REASONS.get(ident), "Provide override reason"

    r = g[g.side == side].iloc[0].to_dict()
    r["selection_reason"] = SIDE_OVERRIDE_REASONS.get(
        ident,
        "left preferred" if side == "L" else "right only; reflected",
    )

    match = photos[photos["individual_id"] == ident]
    if len(match) >= 1:
        r["reference_path"] = match.iloc[0]["reference_path"]
        r["reference_filename"] = match.iloc[0]["reference_filename"]
        r["reference_status"] = "matched"
    else:
        r["reference_path"] = ""
        r["reference_filename"] = ""
        r["reference_status"] = "missing"

    selected.append(r)

selected = (
    pd.DataFrame(selected)
    .sort_values("individual_id", key=lambda s: pd.to_numeric(s))
    .reset_index(drop=True)
)
selected.to_csv(OUTPUT / "selected_source_masks.csv", index=False)

print("Selected masks by side:")
display(selected.groupby("side").size().rename("selected_count"))
print("Reference photo match status:")
display(selected.groupby("reference_status").size().rename("count"))
missing_refs = selected[selected["reference_status"] != "matched"]
if len(missing_refs):
    print("Individuals missing reference-photo matches:")
    display(missing_refs[["individual_id", "side", "filename", "reference_status"]])



## Geometry and automated proposals
Alpha/gamma and delta/epsilon start as silhouette extrema in the Figure 2 orientation; beta is proposed from the deepest hull defect, with a flagged positional fallback. These heuristics do not establish anatomical homology. Preview each grid and correct each point before approval. Automatic batch results stay separate from reviewed results.

Before editing, right masks are reflected, cropped with padding, and optionally rotated. Border contact and discarded foreground are logged. Rotation is applied to the binary working mask; it cannot correct camera perspective.

In [ ]:
def load_working(path,side,angle=0):
    with Image.open(path) as im:
        rgba=im.convert('RGBA')
        gray=np.asarray(Image.alpha_composite(Image.new('RGBA',rgba.size,'white'),rgba).convert('L'))
    binary=gray.astype(np.int16)<255-BACKGROUND_TOLERANCE
    if not binary.any():raise ValueError('No foreground')
    border=bool(binary[0].any() or binary[-1].any() or binary[:,0].any() or binary[:,-1].any())
    lab,n=ndimage.label(binary);counts=np.bincount(lab.ravel());counts[0]=0
    largest=lab==counts.argmax();discarded=float(1-largest.sum()/binary.sum())
    mask=ndimage.binary_fill_holes(largest)
    reflected=side=='R' and not RIGHT_ALREADY_REFLECTED
    if reflected:mask=np.fliplr(mask)
    y,x=np.where(mask);mask=mask[y.min():y.max()+1,x.min():x.max()+1]
    mask=np.pad(mask,max(10,int(.08*max(mask.shape))))
    if angle:mask=ndimage.rotate(mask.astype('uint8'),angle,reshape=True,order=0,cval=0)>0
    contours,_=cv2.findContours(mask.astype('uint8'),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    contour=max(contours,key=cv2.contourArea)
    return mask,contour,{'source_touches_border':border,'discarded_foreground_fraction':discarded,'reflected':reflected,'rotation_degrees':float(angle)}

def propose(contour):
    contour=np.asarray(contour,dtype=np.int32).reshape(-1,1,2)
    pts=contour.reshape(-1,2).astype(float)
    p={'alpha':pts[pts[:,0].argmax()].tolist(),'gamma':pts[pts[:,0].argmin()].tolist(),
       'delta':pts[pts[:,1].argmin()].tolist(),'epsilon':pts[pts[:,1].argmax()].tolist()}
    beta=None
    try:
        hull=cv2.convexHull(contour,returnPoints=False)
        defects=cv2.convexityDefects(contour,hull) if len(hull)>3 else None
        if defects is not None:
           # Handle either (N, 1, 4) or (N, 4) defect arrays.
            defect_rows = np.asarray(defects).reshape(-1, 4)

            if len(defect_rows) > 0:
                defect = defect_rows[np.argmax(defect_rows[:, 3])]

                if defect[3] > 0:
                    beta = pts[int(defect[2])].tolist()
    except cv2.error:pass
    fallback=beta is None
    if fallback:beta=[float(np.mean([p['gamma'][0],p['alpha'][0]])),float(np.mean([p['delta'][1],p['epsilon'][1]]))]
    p['beta']=beta
    return p,fallback

RATIO_PAIRS={'AC_CI':('AC','CI'),'FI_CF':('FI','CF'),'AB_BC':('AB','BC'),'HI_EH':('HI','EH'),'EF_CF':('EF','CF'),'DE_AD':('DE','AD'),'DE_EH':('DE','EH')}
def geometry(p):
    x0,x1,x2=p['gamma'][0],p['beta'][0],p['alpha'][0]
    y0,y1,y2=p['delta'][1],p['beta'][1],p['epsilon'][1]
    if not (x0<x1<x2 and y0<y1<y2):raise ValueError('Require x_gamma < x_beta < x_alpha and y_delta < y_beta < y_epsilon. Check anatomy/orientation, not just numerical order.')
    grid={letter:np.array([x,y],float) for letter,(x,y) in zip('ABCDEFGHI',[(x,y) for y in [y0,y1,y2] for x in [x0,x1,x2]])}
    lengths={s:float(np.linalg.norm(grid[s[1]]-grid[s[0]])) for s in {s for pair in RATIO_PAIRS.values() for s in pair}}
    ratios={name:lengths[a]/lengths[b] for name,(a,b) in RATIO_PAIRS.items()}
    return grid,lengths,ratios

def plot_grid(mask,p,title):
    fig,ax=plt.subplots(figsize=(7,8));ax.imshow(mask,cmap='gray_r')
    try:
        grid,lengths,ratios=geometry(p)
        for a,b in [('A','C'),('D','F'),('G','I'),('A','G'),('B','H'),('C','I')]:
            ax.plot([grid[a][0],grid[b][0]],[grid[a][1],grid[b][1]],color='tab:orange',lw=1)
        for letter,xy in grid.items():ax.annotate(letter,xy,xytext=(3,-8),textcoords='offset points',color='darkred')
    except ValueError as e:ax.text(.01,.01,str(e),transform=ax.transAxes,color='red',wrap=True)
    for label,xy in p.items():ax.plot(*xy,'o',ms=4,color='tab:blue');ax.annotate(label,xy,xytext=(5,8),textcoords='offset points',fontsize=8,color='tab:blue')
    ax.set_title(title);ax.set_aspect('equal');ax.axis('off');fig.tight_layout()
    return fig

# Exact arithmetic check of the grid mapping (not a simulated specimen).
test={'gamma':[0,1],'alpha':[10,2],'beta':[4,3],'delta':[4,0],'epsilon':[6,9]}
g,l,r=geometry(test)
assert np.allclose([r[k] for k in RATIO_PAIRS],[10/9,2,4/6,1,2,4/3,2/3])


In [ ]:
drafts=[];working={};failures=[]
for r in selected.itertuples():
    try:
        mask,contour,info=load_working(r.path,r.side)
        points,fallback=propose(contour)
        working[r.individual_id]={'mask':mask,'points':points,'info':info,'row':r._asdict(),'fallback':fallback}
        record={'individual_id':r.individual_id,'source_side':r.side,'source_filename':r.filename,'status':'unreviewed_proposal',**info,'beta_positional_fallback':fallback}
        try:_,lengths,ratios=geometry(points);record.update(ratios)
        except ValueError as e:record['geometry_error']=str(e)
        drafts.append(record)
        with plt.ioff():
            fig=plot_grid(mask,points,f'{r.individual_id} | source {r.side} | UNREVIEWED')
        fig.savefig(OUTPUT/'draft_overlays'/f'{r.individual_id}.png',dpi=130);plt.close(fig)
    except Exception as e:
        print(f"Failed on skeleton {r.individual_id}: {e}")
        failures.append({'individual_id':r.individual_id,'error':f'{type(e).__name__}: {e}'})

pd.DataFrame(drafts).to_csv(
    OUTPUT / "automatic_proposals_UNREVIEWED.csv",
    index=False,
)

pd.DataFrame(
    failures,
    columns=["individual_id", "error"],
).to_csv(
    OUTPUT / "processing_failures.csv",
    index=False,
)

assert working, "No masks processed"

print(
    f"{len(working)} proposed grids. "
    "Review with the editor below; these are not approved measurements."
)

## Processing check
Do not approve missing or failed proposals. Inspect processing_failures.csv if any failures are reported.

In [ ]:
print(f'Usable proposals: {len(working)}; failures: {len(failures)}')
display(pd.DataFrame(failures,columns=['individual_id','error']))

## Review editor, click landmarks
The masks were processed heuristically but now we need to refine the points, especially beta near the linera inflection. Choose a point and click its location in the left image. On the right, the 2021 annotated trial images appear (might not be accurate), well as the and raw  photos. Turn toolbar pan/zoom off before placement. Drafts persist during navigation but not kernel restart.

In [ ]:
from pathlib import Path
import re
import json
import pandas as pd

if "ROOT" not in globals():
    ROOT = Path.cwd()

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = ROOT / "morphometrix-nonadult" if (ROOT / "morphometrix-nonadult").exists() else ROOT

if "metadata" not in globals():
    metadata_path = ROOT / "consolidated_metadata.csv"
    if not metadata_path.exists():
        metadata_path = PROJECT_ROOT / "consolidated_metadata.csv"
    metadata = pd.read_csv(metadata_path, sep=None, engine="python", dtype=str, keep_default_na=False)
    metadata["individual_id"] = metadata["individual_id"].str.strip().str.replace(",", ".", regex=False)

reference_rows = []
unparsed = []

asset_pattern = re.compile(r'^(?P<id>\d+(?:[.,-]\d+)?)(?P<side>[LR])?$', re.I)
raw_pattern = re.compile(r'^f(?P<id>\d+(?:[.,-]\d+)?)P.+$', re.I)

for folder, kind, pattern in [
    (PROJECT_ROOT / "bones-assets", "Photoshop grid", asset_pattern),
    (PROJECT_ROOT / "f", "Raw photo", raw_pattern),
]:
    if not folder.is_dir():
        print(f"Optional reference folder not found: {folder}")
        continue

    for path in sorted(folder.rglob("*"), key=lambda p: str(p).lower()):
        if path.suffix.lower() not in {".png", ".jpg", ".jpeg", ".tif", ".tiff"}:
            continue

        match = pattern.match(path.stem)
        if match is None:
            unparsed.append({"filename": path.name, "folder": folder.name})
            continue

        reference_rows.append(
            {
                "individual_id": match["id"].replace(",", ".").replace("-", "."),
                "reference_type": kind,
                "explicit_side": (match.groupdict().get("side") or "").upper(),
                "filename": path.name,
                "relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
                "absolute_path": str(path.resolve()),
            }
        )

reference_lookup = {}
for row in reference_rows:
    reference_lookup.setdefault(row["individual_id"], []).append(row)

print(f"{len(reference_rows)} references loaded.")
print(f"{len(reference_lookup)} individuals with at least one reference.")
if unparsed:
    print(f"{len(unparsed)} reference files did not match the filename patterns.")

In [ ]:
import copy
import hashlib
import html
import io
import json
import uuid
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image, ImageOps

# ----------------------------
# Safe fallbacks for cell-order robustness
# ----------------------------
if "ROOT" not in globals():
    ROOT = Path.cwd()

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = ROOT / "morphometrix-nonadult" if (ROOT / "morphometrix-nonadult").exists() else ROOT

if "OUTPUT" not in globals():
    OUTPUT = PROJECT_ROOT / "luna_output"

if "reference_lookup" not in globals():
    reference_lookup = {}

if "photo_state" not in globals():
    photo_state = {"loading": False, "records": {}}

if "saved_this_session" not in globals():
    saved_this_session = set()

if "editor_drafts" not in globals():
    editor_drafts = {}

if "OBSERVER_ID" not in globals():
    OBSERVER_ID = "observer_1"

if "SESSION_ID" not in globals():
    SESSION_ID = "session_1"

if "BACKGROUND_TOLERANCE" not in globals():
    BACKGROUND_TOLERANCE = 20

if "RIGHT_ALREADY_REFLECTED" not in globals():
    RIGHT_ALREADY_REFLECTED = False

if "working" not in globals():
    raise RuntimeError("`working` is not defined. Run the earlier setup / proposal cells first.")

if "plot_grid" not in globals():
    raise RuntimeError("`plot_grid` is not defined. Run the geometry/plotting cell first.")

if "geometry" not in globals():
    raise RuntimeError("`geometry` is not defined. Run the geometry cell first.")

if "load_working" not in globals():
    raise RuntimeError("`load_working` is not defined. Run the preprocessing cell first.")

if "propose" not in globals():
    raise RuntimeError("`propose` is not defined. Run the proposal cell first.")

if "RATIO_PAIRS" not in globals():
    raise RuntimeError("`RATIO_PAIRS` is not defined. Run the geometry cell first.")

for folder in ["reviewed_overlays", "annotations"]:
    (OUTPUT / folder).mkdir(parents=True, exist_ok=True)

previous_editor_state = globals().get("editor_state", {})
if previous_editor_state.get("fig") is not None:
    plt.close(previous_editor_state["fig"])

ids = sorted(working, key=float)
assert ids, "No working proposals available."

LANDMARKS = ["beta", "alpha", "gamma", "delta", "epsilon"]

previous_btn = widgets.Button(layout=widgets.Layout(width="100%"))
next_btn = widgets.Button(layout=widgets.Layout(width="100%"))

picker = widgets.Dropdown(
    options=ids,
    description="Individual:",
    layout=widgets.Layout(width="100%"),
)

landmark_picker = widgets.ToggleButtons(
    options=LANDMARKS,
    value="beta",
    description="Place:",
)

auto_advance = widgets.Checkbox(
    value=True,
    description="Advance to next landmark after clicking",
    indent=False,
)

angle = widgets.FloatText(
    value=0.0,
    description="Mask rot.°",
    layout=widgets.Layout(width="170px"),
)

rotate_btn = widgets.Button(
    description="Apply mask rotation (reset proposals)",
    layout=widgets.Layout(width="280px"),
)

ref_angle = widgets.FloatText(
    value=0.0,
    description="Ref rot.°",
    layout=widgets.Layout(width="170px"),
)

ref_rotate_btn = widgets.Button(
    description="Apply ref rotation",
    layout=widgets.Layout(width="180px"),
)

photo_heading = widgets.HTML("<h4>Reference picture</h4>")

photo_picker = widgets.Dropdown(
    description="Reference:",
    options=[],
    layout=widgets.Layout(width="100%"),
    style={"description_width": "initial"},
)

photo_previous = widgets.Button(
    description="Previous reference",
    layout=widgets.Layout(width="49%"),
)

photo_next = widgets.Button(
    description="Next reference",
    layout=widgets.Layout(width="49%"),
)

photo_size = widgets.IntSlider(
    value=1200,
    min=500,
    max=2200,
    step=100,
    description="Preview pixels:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%"),
)

photo_image = widgets.Image(
    format="png",
    layout=widgets.Layout(width="100%", height="auto"),
)

photo_caption = widgets.HTML()

preview_btn = widgets.Button(description="Refresh grid")
save_btn = widgets.Button(
    description="Snapshot save",
    button_style="success",
    layout=widgets.Layout(width="180px"),
)

notes = widgets.Textarea(
    description="Notes:",
    layout=widgets.Layout(width="95%", height="80px"),
)

heading = widgets.HTML()
instruction = widgets.HTML()
view = widgets.Output()
ratios_view = widgets.Output()
message = widgets.Output()

editor_state = {
    "id": None,
    "data": None,
    "fig": None,
    "ax": None,
    "loading": False,
    "dirty": False,
}

def notify(text):
    with message:
        clear_output(wait=True)
        print(text)

def filename_for(ident):
    row = working[ident]["row"]
    return str(row.get("filename", Path(row["path"]).name))

def destination_label(ident, direction):
    return f"{direction}: {ident} — {filename_for(ident)}"

def update_navigation():
    index = ids.index(picker.value)
    previous_btn.disabled = index == 0
    next_btn.disabled = index == len(ids) - 1

    previous_btn.description = (
        destination_label(ids[index - 1], "Previous")
        if index > 0 else
        "Previous: start of list"
    )

    next_btn.description = (
        destination_label(ids[index + 1], "Next")
        if index < len(ids) - 1 else
        "Next: end of list"
    )

    previous_btn.tooltip = previous_btn.description
    next_btn.tooltip = next_btn.description

def update_heading():
    ident = editor_state["id"]
    row = editor_state["data"]["row"]
    filename = str(row.get("filename", Path(row["path"]).name))
    saved = " | saved this session" if ident in saved_this_session else ""
    dirty = " | unsaved changes" if editor_state.get("dirty") else ""

    heading.value = (
        f"<h3>Individual {html.escape(str(ident))} — {html.escape(filename)}</h3>"
        f"<div>Source side: {html.escape(str(row['side']))} "
        f"| {ids.index(ident) + 1} of {len(ids)}{saved}{dirty}</div>"
    )

def update_instruction(*_):
    instruction.value = (
        "Select a landmark, then left-click its location. "
        f"Currently placing: <b>{landmark_picker.value}</b>. "
        "Turn off the plot toolbar's pan/zoom tool before placing points."
    )

def cache_current():
    if editor_state["id"] is None:
        return
    editor_drafts[editor_state["id"]] = {
        "data": copy.deepcopy(editor_state["data"]),
        "notes": notes.value,
    }

def points_from_editor():
    return {
        name: [
            float(editor_state["data"]["points"][name][0]),
            float(editor_state["data"]["points"][name][1]),
        ]
        for name in LANDMARKS
    }

def current_row():
    return editor_state["data"]["row"]

def latest_json_path(ident):
    return OUTPUT / "annotations" / f"{ident}_latest.json"

def latest_overlay_path(ident):
    return OUTPUT / "reviewed_overlays" / f"{ident}_latest.png"

def current_editor_side():
    data = editor_state.get("data")
    return str(data["row"].get("side", "")).upper() if data is not None else ""

def reference_priority(record, side):
    if record["reference_type"] == "Raw photo":
        rank = 3
    elif side and record.get("explicit_side", "") == side:
        rank = 0
    elif not record.get("explicit_side", ""):
        rank = 1
    else:
        rank = 2
    return rank, record["filename"].lower()

def display_reference_photo(change=None):
    if photo_state["loading"]:
        return

    values = [v for label, v in photo_picker.options]

    if not values or photo_picker.value is None:
        photo_image.value = b""
        photo_caption.value = (
            "No linked reference image. Run the reference-discovery cell if you want photo browsing."
        )
        photo_previous.disabled = True
        photo_next.disabled = True
        return

    selected = photo_picker.value
    index = values.index(selected)
    record = photo_state["records"][selected]

    photo_previous.disabled = index == 0
    photo_next.disabled = index == len(values) - 1

    try:
        ref_path = PROJECT_ROOT / selected if not Path(selected).is_absolute() else Path(selected)

        with Image.open(ref_path) as source:
            rgba = ImageOps.exif_transpose(source).convert("RGBA")
            original_size = rgba.size

            rotation = float(editor_state["data"]["info"].get("reference_rotation_degrees", 0.0))
            if rotation != 0:
                rgba = rgba.rotate(
                    -rotation,
                    expand=True,
                    resample=Image.Resampling.BICUBIC,
                )

            preview = Image.alpha_composite(
                Image.new("RGBA", rgba.size, "white"),
                rgba,
            ).convert("RGB")

            preview.thumbnail(
                (photo_size.value, photo_size.value),
                Image.Resampling.LANCZOS,
            )

            buffer = io.BytesIO()
            preview.save(buffer, format="PNG")

        photo_image.value = buffer.getvalue()

        notice = ""
        if (
            record.get("explicit_side")
            and current_editor_side()
            and record["explicit_side"] != current_editor_side()
        ):
            notice = (
                "<div><b>Opposite-side reference</b>: "
                "check anatomy; this reference is not mirrored.</div>"
            )

        photo_caption.value = (
            f"<div><b>{html.escape(record['reference_type'])}</b> | "
            f"{html.escape(record['filename'])} | "
            f"{index + 1}/{len(values)} | "
            f"original {original_size[0]} × {original_size[1]} px</div>"
            + notice +
            "<div>Reference only: not registered to the mask. "
            "Place landmarks in the editor, not on this photo.</div>"
        )

    except Exception as exc:
        photo_image.value = b""
        photo_caption.value = html.escape(
            f"Could not display image: {type(exc).__name__}: {exc}"
        )

def load_reference_photos(ident):
    records = sorted(
        reference_lookup.get(str(ident), []),
        key=lambda r: reference_priority(r, current_editor_side()),
    )

    photo_state["loading"] = True
    try:
        photo_state["records"] = {
            (r["relative_path"] if "relative_path" in r else r.get("absolute_path", "")): r
            for r in records
        }
        photo_heading.value = f"<h4>References — individual {html.escape(str(ident))}</h4>"
        photo_picker.options = [
            (
                f"{r['reference_type']} | {r['filename']}"
                + (f" | side {r.get('explicit_side', '')}" if r.get("explicit_side") else ""),
                r["relative_path"] if "relative_path" in r else r.get("absolute_path", ""),
            )
            for r in records
        ]
        photo_picker.value = photo_picker.options[0][1] if photo_picker.options else None
    finally:
        photo_state["loading"] = False

    display_reference_photo()

def step_reference_photo(offset):
    values = [v for label, v in photo_picker.options]
    if photo_picker.value not in values:
        return
    target = values.index(photo_picker.value) + offset
    if 0 <= target < len(values):
        photo_picker.value = values[target]

def mark_dirty():
    editor_state["dirty"] = True
    update_heading()

def preview(_=None):
    old_fig = editor_state.get("fig")
    if old_fig is not None:
        plt.close(old_fig)

    data = editor_state["data"]
    points = points_from_editor()
    ident = editor_state["id"]

    with view:
        clear_output(wait=True)
        with plt.ioff():
            fig = plot_grid(data["mask"], points, f"{ident} source {data['row']['side']} — review draft")
            ax = fig.axes[0]
            ax.axis("on")

            for name, (x, y) in points.items():
                selected_here = name == landmark_picker.value
                ax.scatter(
                    x,
                    y,
                    s=85 if selected_here else 40,
                    c="yellow" if selected_here else "red",
                    edgecolors="black",
                    zorder=20,
                )
                ax.annotate(
                    name,
                    (x, y),
                    xytext=(7, 7),
                    textcoords="offset points",
                    fontsize=9,
                    color="black",
                    bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"),
                    zorder=21,
                )

            editor_state["fig"] = fig
            editor_state["ax"] = ax
            fig.canvas.mpl_connect("button_press_event", on_image_click)
            display(fig.canvas)

    with ratios_view:
        clear_output(wait=True)
        try:
            display(pd.Series(geometry(points)[2], name="Current ratios"))
        except Exception as exc:
            print("Grid geometry is not valid yet. Adjust the landmarks.")
            print(f"{type(exc).__name__}: {exc}")

    display_reference_photo()
    update_heading()
    update_instruction()

def on_image_click(event):
    ax = editor_state.get("ax")
    if event.inaxes is not ax:
        return
    if event.button != 1:
        return
    if event.xdata is None or event.ydata is None:
        return

    toolbar = getattr(event.canvas, "toolbar", None)
    if toolbar is not None and getattr(toolbar, "mode", ""):
        return

    x = float(event.xdata)
    y = float(event.ydata)
    height, width = editor_state["data"]["mask"].shape
    if not (0 <= x < width and 0 <= y < height):
        return

    name = landmark_picker.value
    editor_state["data"]["points"][name] = [x, y]

    if auto_advance.value:
        index = LANDMARKS.index(name)
        if index < len(LANDMARKS) - 1:
            landmark_picker.value = LANDMARKS[index + 1]

    mark_dirty()
    cache_current()
    preview()
    save(silent=True, snapshot=False)
    notify(f"Placed {name} at x={x:.1f}, y={y:.1f}. Saved latest annotation.")

def load_individual(change=None):
    if editor_state["loading"]:
        return

    ident = picker.value
    editor_state["loading"] = True
    try:
        if ident in editor_drafts:
            draft = editor_drafts[ident]
            editor_state["data"] = copy.deepcopy(draft["data"])
            notes.value = draft["notes"]
        else:
            editor_state["data"] = copy.deepcopy(working[ident])
            notes.value = ""

        editor_state["id"] = ident
        editor_state["dirty"] = False
        angle.value = float(editor_state["data"]["info"].get("rotation_degrees", 0.0))
        ref_angle.value = float(editor_state["data"]["info"].get("reference_rotation_degrees", 0.0))
        landmark_picker.value = "beta"
    finally:
        editor_state["loading"] = False

    update_navigation()
    load_reference_photos(ident)
    preview()

    notify(
        "Individual loaded. Clicking places points and saves automatically. "
        "Previous/Next also save changes automatically."
    )

def maybe_save_before_navigation():
    cache_current()
    if editor_state.get("dirty"):
        save(silent=True, snapshot=False)

def previous_individual(_):
    maybe_save_before_navigation()
    index = ids.index(picker.value)
    if index > 0:
        picker.value = ids[index - 1]

def next_individual(_):
    maybe_save_before_navigation()
    index = ids.index(picker.value)
    if index < len(ids) - 1:
        picker.value = ids[index + 1]

def apply_rotation(_):
    try:
        data = editor_state["data"]
        row = data["row"]
        mask, contour, info = load_working(row["path"], row["side"], angle.value)
        info["reference_rotation_degrees"] = float(ref_angle.value)
        points, fallback = propose(contour)

        editor_state["data"] = {
            "mask": mask,
            "points": points,
            "info": info,
            "row": row,
            "fallback": fallback,
        }

        landmark_picker.value = "beta"
        mark_dirty()
        cache_current()
        load_reference_photos(editor_state["id"])
        preview()
        save(silent=True, snapshot=False)

        notify(
            "Mask rotation applied. Landmarks were reset to new automatic proposals and saved. "
            "Review all five points again."
        )
    except Exception as exc:
        notify(f"Rotation failed: {type(exc).__name__}: {exc}")

def apply_reference_rotation(_):
    try:
        editor_state["data"]["info"]["reference_rotation_degrees"] = float(ref_angle.value)
        mark_dirty()
        cache_current()
        display_reference_photo()
        save(silent=True, snapshot=False)
        notify(f"Reference rotation updated to {ref_angle.value:.1f}° and saved.")
    except Exception as exc:
        notify(f"Reference rotation failed: {type(exc).__name__}: {exc}")

def save(_=None, silent=False, snapshot=True):
    try:
        if not OBSERVER_ID or not SESSION_ID:
            raise ValueError("Observer and session identifiers are required.")

        data = editor_state["data"]
        points = points_from_editor()

        for xy in points.values():
            if not np.isfinite(xy).all():
                raise ValueError("Landmarks must have finite coordinates.")

        grid, lengths, ratios = geometry(points)
        height, width = data["mask"].shape

        if not all(0 <= xy[0] < width and 0 <= xy[1] < height for xy in points.values()):
            raise ValueError("Points are outside the working image.")

        row = data["row"]
        ident = editor_state["id"]
        revision = uuid.uuid4().hex

        record = {
            **data["info"],
            "individual_id": ident,
            "observer_id": OBSERVER_ID,
            "session_id": SESSION_ID,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "revision": revision,
            "source_side": row["side"],
            "source_filename": str(row.get("filename", Path(row["path"]).name)),
            "selection_reason": row.get("selection_reason", ""),
            "source_sha256": hashlib.sha256(Path(row["path"]).read_bytes()).hexdigest(),
            "status": "autosaved_latest" if not snapshot else "manual_snapshot",
            "notes": notes.value,
            "background_tolerance": BACKGROUND_TOLERANCE,
            "right_already_reflected_setting": RIGHT_ALREADY_REFLECTED,
            "working_shape": list(data["mask"].shape),
            "beta_proposal_fallback": bool(data["fallback"]),
            "landmarks": points,
            "grid": {key: np.asarray(value).tolist() for key, value in grid.items()},
            "lengths_px": lengths,
            "ratios": ratios,
        }

        def json_default(value):
            if isinstance(value, np.ndarray):
                return value.tolist()
            if isinstance(value, np.generic):
                return value.item()
            raise TypeError(f"Cannot serialize type {type(value).__name__}")

        payload = json.dumps(record, indent=2, default=json_default, allow_nan=False)

        with plt.ioff():
            fig = plot_grid(
                data["mask"],
                points,
                f"{ident} {OBSERVER_ID}/{SESSION_ID} {'snapshot' if snapshot else 'latest'}"
            )
            try:
                fig.savefig(latest_overlay_path(ident), dpi=160)
                if snapshot:
                    fig.savefig(OUTPUT / "reviewed_overlays" / f"{ident}_{revision}.png", dpi=160)
            finally:
                plt.close(fig)

        latest_json_path(ident).write_text(payload, encoding="utf-8")
        if snapshot:
            (OUTPUT / "annotations" / f"{ident}_{revision}.json").write_text(payload, encoding="utf-8")

        cache_current()
        editor_state["dirty"] = False
        saved_this_session.add(ident)
        update_heading()

        if not silent:
            if snapshot:
                notify(f"Snapshot saved for individual {ident}. Latest autosave file was also updated.")
            else:
                notify(f"Saved latest annotation for individual {ident}.")
    except Exception as exc:
        notify(f"Not saved: {type(exc).__name__}: {exc}")

picker.observe(load_individual, names="value")
previous_btn.on_click(previous_individual)
next_btn.on_click(next_individual)
landmark_picker.observe(update_instruction, names="value")
photo_picker.observe(display_reference_photo, names="value")
photo_size.observe(display_reference_photo, names="value")
photo_previous.on_click(lambda _: step_reference_photo(-1))
photo_next.on_click(lambda _: step_reference_photo(1))
preview_btn.on_click(preview)
rotate_btn.on_click(apply_rotation)
ref_rotate_btn.on_click(apply_reference_rotation)
save_btn.on_click(lambda _: save(snapshot=True))

left_panel = widgets.VBox(
    [
        heading,
        widgets.HBox([angle, rotate_btn]),
        landmark_picker,
        auto_advance,
        instruction,
        view,
        ratios_view,
        notes,
        widgets.HBox([preview_btn, save_btn]),
        message,
    ],
    layout=widgets.Layout(width="50%", min_width="560px"),
)

right_panel = widgets.VBox(
    [
        previous_btn,
        next_btn,
        picker,
        photo_heading,
        photo_picker,
        widgets.HBox([photo_previous, photo_next]),
        widgets.HBox([ref_angle, ref_rotate_btn]),
        photo_size,
        photo_image,
        photo_caption,
    ],
    layout=widgets.Layout(width="50%", min_width="560px"),
)

display(
    widgets.HBox(
        [left_panel, right_panel],
        layout=widgets.Layout(
            width="100%",
            align_items="flex-start",
            justify_content="space-between",
            gap="5px",
        ),
    )
)

load_individual()

## Export current observer/session reviewed measurements
Run whenever you want updated CSVs. Keeps one latest approved revision per individual/observer/session, all revision history stays on disk. All metadata and canine fields are joined unchanged. Review status includes individuals not yet annotated. Raw segment lengths are working pixels, ratios dimensionless. Right-derived measurements are identifiable and should be included in a side-sensitivity analysis rather than claimed as exact replication of the left-only original study.

In [ ]:
annotations=[]
for p in (OUTPUT/'annotations').glob('*.json'):
    rec=json.loads(p.read_text())
    if rec['observer_id']!=OBSERVER_ID or rec['session_id']!=SESSION_ID:continue
    flat={k:v for k,v in rec.items() if k not in {'landmarks','grid','lengths_px','ratios','working_shape'}}
    flat.update({f'{name}_{axis}':xy[j] for name,xy in rec['landmarks'].items() for j,axis in enumerate(['x','y'])})
    flat.update({f'{s}_px':v for s,v in rec['lengths_px'].items()});flat.update(rec['ratios'])
    annotations.append(flat)
if annotations:
    reviewed=pd.DataFrame(annotations).sort_values('timestamp_utc').drop_duplicates(['individual_id','observer_id','session_id'],keep='last')
    reviewed=reviewed.sort_values('individual_id',key=lambda s:pd.to_numeric(s))
    tag=re.sub(r'[^A-Za-z0-9_.-]','_',f'{OBSERVER_ID}_{SESSION_ID}')
    reviewed.to_csv(OUTPUT/f'luna_ratios_reviewed_{tag}.csv',index=False)
    joined=metadata.merge(reviewed,on='individual_id',how='left',validate='one_to_one',suffixes=('','_luna')).sort_values('individual_id',key=lambda s:pd.to_numeric(s))
    joined.to_csv(OUTPUT/f'luna_ratios_metadata_canines_{tag}.csv',index=False)
    status=selected[['individual_id','side','filename']].copy();status['approved_this_session']=status.individual_id.isin(reviewed.individual_id)
    status.to_csv(OUTPUT/f'review_progress_{tag}.csv',index=False)
    display(reviewed[['individual_id','source_side']+list(RATIO_PAIRS)])
    print(f'{len(reviewed)} reviewed of {len(selected)} selected individuals')
else:print('No approved annotations saved for current observer/session yet.')


Exporting the data

In [ ]:
import json
import re
import numpy as np
import pandas as pd

all_rows = []

tag = re.sub(r"[^A-Za-z0-9_.-]", "_", f"{OBSERVER_ID}_{SESSION_ID}")

latest_manual = {}
for p in (OUTPUT / "annotations").glob("*_latest.json"):
    rec = json.loads(p.read_text(encoding="utf-8"))
    if rec.get("observer_id") != OBSERVER_ID or rec.get("session_id") != SESSION_ID:
        continue
    latest_manual[str(rec["individual_id"])] = rec

for ident in sorted(selected["individual_id"].astype(str), key=float):
    row = selected[selected["individual_id"].astype(str) == ident].iloc[0].to_dict()

    if ident in latest_manual:
        rec = latest_manual[ident]

        flat = {
            k: v for k, v in rec.items()
            if k not in {"landmarks", "grid", "lengths_px", "ratios", "working_shape"}
        }

        flat.update({
            f"{name}_{axis}": xy[j]
            for name, xy in rec["landmarks"].items()
            for j, axis in enumerate(["x", "y"])
        })

        flat.update({f"{seg}_px": value for seg, value in rec["lengths_px"].items()})
        flat.update(rec["ratios"])

        flat["measurement_source"] = "manual_or_reviewed_latest"
        flat["was_reviewed_this_session"] = True

    else:
        item = working[ident]
        points = item["points"]
        grid, lengths, ratios = geometry(points)
        info = item["info"]

        flat = {
            "individual_id": ident,
            "observer_id": OBSERVER_ID,
            "session_id": SESSION_ID,
            "timestamp_utc": "",
            "revision": "",
            "source_side": row["side"],
            "source_filename": str(row.get("filename", Path(row["path"]).name)),
            "selection_reason": row.get("selection_reason", ""),
            "status": "automatic_unedited",
            "notes": "",
            "background_tolerance": BACKGROUND_TOLERANCE,
            "right_already_reflected_setting": RIGHT_ALREADY_REFLECTED,
            "working_shape": list(item["mask"].shape),
            "beta_proposal_fallback": bool(item["fallback"]),
            "source_touches_border": info.get("source_touches_border"),
            "discarded_foreground_fraction": info.get("discarded_foreground_fraction"),
            "reflected": info.get("reflected"),
            "rotation_degrees": info.get("rotation_degrees"),
        }

        flat.update({
            f"{name}_{axis}": points[name][j]
            for name in LANDMARKS
            for j, axis in enumerate(["x", "y"])
        })

        flat.update({f"{seg}_px": value for seg, value in lengths.items()})
        flat.update(ratios)

        flat["measurement_source"] = "automatic_working"
        flat["was_reviewed_this_session"] = False

    all_rows.append(flat)

all_measurements = pd.DataFrame(all_rows).sort_values(
    "individual_id", key=lambda s: pd.to_numeric(s)
)

all_measurements.to_csv(
    OUTPUT / f"luna_measurements_all_selected_{tag}.csv",
    index=False,
)

joined = metadata.merge(
    all_measurements,
    on="individual_id",
    how="right",
    validate="one_to_one",
    suffixes=("", "_luna"),
).sort_values("individual_id", key=lambda s: pd.to_numeric(s))

joined.to_csv(
    OUTPUT / f"luna_measurements_all_selected_with_metadata_{tag}.csv",
    index=False,
)

progress = selected[["individual_id", "side", "filename"]].copy()
progress["has_measurement"] = progress["individual_id"].astype(str).isin(all_measurements["individual_id"].astype(str))
progress["was_reviewed_this_session"] = progress["individual_id"].astype(str).isin(latest_manual.keys())
progress["measurement_source"] = progress["individual_id"].astype(str).map(
    dict(zip(all_measurements["individual_id"].astype(str), all_measurements["measurement_source"]))
)

progress.to_csv(
    OUTPUT / f"measurement_progress_{tag}.csv",
    index=False,
)

display(
    all_measurements[
        ["individual_id", "source_side", "measurement_source", "FI_CF", "DE_AD"]
    ].head(20)
)

print(f"{len(all_measurements)} measurements exported for {len(selected)} selected individuals.")
print(f"{sum(all_measurements['measurement_source'] == 'manual_or_reviewed_latest')} reviewed manually in this session.")
print(f"{sum(all_measurements['measurement_source'] == 'automatic_working')} retained as automatic placements.")

# Fishergate auricular surface analysis. transfer formulas, clustering, and visualisation

Applying the formulas from Luna et al. 2017. Because Fishergate House does not have documented sex, we will experimentally use the Coimbra equations and section points, but only as published external-transfer tools. One should not assume they are valid Fishergate thresholds, and should not recalculate or claim Fishergate accuracy without more reliable sex estimation labels.


## Analysis design

three parallel outputs throughout the notebook:

1. **Coimbra thresholds and formulas** applied unchanged as external-transfer results.
2. **Fishergate exploratory thresholds** estimated only as a sensitivity analysis when independent sex estimates exist.
3. **Unsupervised shape structure** using PCA and clustering without sex labels.

The first stream is the primary analysis. The second is clearly secondary. The third may become the most informative stream if sex labels are sparse or uncertain.


## Original paper reference points

Luna, Aranda and Santos (2017) defined seven ratios from the auricular-surface grid and found FI/CF and DE/AD to be the strongest dimorphic predictors in the Coimbra non-adult sample. They reported a discriminant function and a logistic regression using those two ratios. This notebook applies those formulas unchanged as transfer models and keeps them analytically separate from any Fishergate-derived exploratory analyses.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    from sklearn.cluster import AgglomerativeClustering, KMeans
    from sklearn.metrics import silhouette_score
    SKLEARN_OK = True
except ModuleNotFoundError:
    SKLEARN_OK = False
    print('scikit-learn is not installed. Run %pip install scikit-learn and restart the kernel for clustering/PCA cells.')

ROOT = Path.cwd()
OUT = ROOT / 'analysis_output'
OUT.mkdir(exist_ok=True, parents=True)
CSV_NAME = 'luna_measurements_all_selected_with_metadata_observer_1_session_1.csv'
candidate_paths = [ROOT / CSV_NAME, ROOT / 'luna_output' / CSV_NAME]
csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(f'Could not locate {CSV_NAME} in the working directory or luna_output/.')
print('Loading:', csv_path)


In [ ]:
df = pd.read_csv(csv_path, dtype={'individual_id': str})
df['individual_id'] = df['individual_id'].astype(str).str.strip()
print('Rows, columns:', df.shape)
display(df.head(5))


In [ ]:
ratio_cols = ['AC_CI', 'FI_CF', 'AB_BC', 'HI_EH', 'EF_CF', 'DE_AD', 'DE_EH']
length_cols = ['CF_px', 'DE_px', 'AB_px', 'AC_px', 'EH_px', 'AD_px', 'CI_px', 'BC_px', 'HI_px', 'EF_px', 'FI_px']
landmark_cols = ['beta_x', 'beta_y', 'alpha_x', 'alpha_y', 'gamma_x', 'gamma_y', 'delta_x', 'delta_y', 'epsilon_x', 'epsilon_y']
preferred_meta = ['individual_id', 'sex_code', 'age_original', 'age_group', 'preservation', 'sides_available', 'source_side', 'measurement_source', 'was_reviewed_this_session', 'selection_reason', 'source_filename']
slim_cols = [c for c in preferred_meta + ratio_cols + length_cols + landmark_cols if c in df.columns]
slim_df = df[slim_cols].copy()
for c in ratio_cols + length_cols + landmark_cols:
    if c in slim_df.columns:
        slim_df[c] = pd.to_numeric(slim_df[c], errors='coerce')
for c in ['source_side', 'sex_code', 'measurement_source', 'age_original', 'age_group']:
    if c in slim_df.columns:
        slim_df[c] = slim_df[c].astype(str).str.strip()
slim_df.to_csv(OUT / 'fishergate_luna_analysis_slim.csv', index=False)
print('Saved slim export:', OUT / 'fishergate_luna_analysis_slim.csv')
display(slim_df.head(5))


## Coimbra transfer formulas

Original discriminant function reported by Luna et al. (2017):

`X = -3.470 + 0.865*(DE/AD) + 2.739*(FI/CF)`

Section point: `0.078`

Original logistic model reported by Luna et al. (2017):

`P(male) = 1 / (1 + exp(-(-25.819 + 37.723*(DE/AD) - 2.320*(FI/CF))))`

These formulas were developed on a documented Coimbra sample aged 7–18 years. Here they are used as transfer models, not recalibrated Fishergate models.


In [ ]:
formula_df = slim_df[['individual_id'] + [c for c in ['sex_code', 'age_original', 'age_group', 'source_side', 'measurement_source'] if c in slim_df.columns] + ['FI_CF', 'DE_AD']].copy()
formula_df = formula_df.dropna(subset=['FI_CF', 'DE_AD']).reset_index(drop=True)
formula_df['coimbra_discriminant_score'] = -3.470 + 0.865 * formula_df['DE_AD'] + 2.739 * formula_df['FI_CF']
formula_df['coimbra_discriminant_pred'] = np.where(formula_df['coimbra_discriminant_score'] > 0.078, 'M', 'F')
formula_df['coimbra_logit_linear'] = -25.819 + 37.723 * formula_df['DE_AD'] - 2.320 * formula_df['FI_CF']
formula_df['coimbra_logit_probability_male'] = 1 / (1 + np.exp(-formula_df['coimbra_logit_linear']))
formula_df['coimbra_logit_pred'] = np.where(formula_df['coimbra_logit_probability_male'] >= 0.5, 'M', 'F')
formula_df.to_csv(OUT / 'fishergate_coimbra_transfer_predictions.csv', index=False)
display(formula_df.head(10))


In [ ]:
def age_midpoint(value):
    s = str(value).strip().upper()
    if not s or s == 'NAN':
        return np.nan
    nums = [float(x) for x in re.findall(r'\d+(?:\.\d+)?', s)]
    if not nums:
        return np.nan
    if 'W' in s:
        return nums[0] / 52.0
    if 'M' in s and 'W' not in s:
        return np.mean(nums) / 12.0
    return float(np.mean(nums))

formula_df['age_midpoint_years'] = formula_df['age_original'].map(age_midpoint) if 'age_original' in formula_df.columns else np.nan
formula_df['within_luna_7_18'] = formula_df['age_midpoint_years'].between(7, 18, inclusive='both')
display(formula_df[['individual_id', 'age_original', 'age_midpoint_years', 'within_luna_7_18']].head(10))
display(formula_df['within_luna_7_18'].value_counts(dropna=False))


## Fishergate exploratory thresholds

This section estimates midpoint thresholds for FI/CF and DE/AD only when both `M` and `F` labels are present in the current data. Using 4 individuals only:

57	M	17-19, 209	M	17-25

142	F	16-17, 173	F	17-19 

Treat these results as sensitivity checks, not as primary inferential models.


In [ ]:
explore_df = formula_df.copy()
if 'sex_code' in explore_df.columns:
    explore_df['sex_code'] = explore_df['sex_code'].astype(str).str.upper().str.strip()
    explore_df = explore_df[explore_df['sex_code'].isin(['M', 'F'])].copy()

if len(explore_df) > 0 and explore_df['sex_code'].nunique() == 2:
    rows = []
    for var in ['FI_CF', 'DE_AD']:
        means = explore_df.groupby('sex_code')[var].mean()
        rows.append({
            'variable': var,
            'female_mean': means.get('F', np.nan),
            'male_mean': means.get('M', np.nan),
            'midpoint_threshold': np.mean([means.get('F', np.nan), means.get('M', np.nan)])
        })
    thresholds_df = pd.DataFrame(rows)
    thresholds_df.to_csv(OUT / 'fishergate_exploratory_thresholds.csv', index=False)
    display(thresholds_df)
else:
    print('Skipped Fishergate exploratory thresholds: insufficient M/F labels.')


## Unsupervised shape structure

The seven Luna ratios are used as the primary exploratory feature space. PCA is used for visualization and dimensional reduction; k-means and Ward clustering are applied only if scikit-learn is installed.


In [ ]:
feature_cols = [c for c in ratio_cols if c in slim_df.columns]
analysis = slim_df[['individual_id'] + [c for c in preferred_meta if c in slim_df.columns and c != 'individual_id'] + feature_cols].copy()
analysis = analysis.dropna(subset=feature_cols).reset_index(drop=True)
print('Rows with complete ratio data:', len(analysis))


In [ ]:
if not SKLEARN_OK:
    print('Install scikit-learn to run PCA and clustering.')
else:
    X = analysis[feature_cols].to_numpy(dtype=float)
    Xz = StandardScaler().fit_transform(X)
    pca = PCA()
    Xp = pca.fit_transform(Xz)
    pc_names = [f'PC{i+1}' for i in range(Xp.shape[1])]
    analysis_pca = pd.concat([analysis.reset_index(drop=True), pd.DataFrame(Xp, columns=pc_names)], axis=1)
    analysis_pca['age_midpoint_years'] = analysis_pca['age_original'].map(age_midpoint) if 'age_original' in analysis_pca.columns else np.nan
    analysis_pca.to_csv(OUT / 'fishergate_luna_analysis_pca_scores.csv', index=False)
    pd.DataFrame({'component': pc_names, 'explained_variance_ratio': pca.explained_variance_ratio_, 'cumulative_variance_ratio': np.cumsum(pca.explained_variance_ratio_)}).to_csv(OUT / 'fishergate_luna_pca_variance.csv', index=False)
    display(analysis_pca.head(5))


In [ ]:
if not SKLEARN_OK:
    print('Install scikit-learn to run clustering.')
else:
    scores = []
    for k in range(2, min(7, len(analysis))):
        labels = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(Xz)
        scores.append({'k': k, 'silhouette': silhouette_score(Xz, labels)})
    scores_df = pd.DataFrame(scores)
    scores_df.to_csv(OUT / 'fishergate_luna_cluster_silhouette.csv', index=False)
    best_k = int(scores_df.sort_values(['silhouette', 'k'], ascending=[False, True]).iloc[0]['k'])
    analysis_pca['cluster_kmeans'] = KMeans(n_clusters=best_k, n_init=50, random_state=42).fit_predict(Xz).astype(int)
    analysis_pca['cluster_ward'] = AgglomerativeClustering(n_clusters=best_k, linkage='ward').fit_predict(Xz).astype(int)
    analysis_pca.to_csv(OUT / 'fishergate_luna_clustered.csv', index=False)
    analysis_pca.groupby('cluster_kmeans')[feature_cols].agg(['mean', 'median', 'count']).to_csv(OUT / 'fishergate_luna_cluster_summary.csv')
    display(scores_df)
    display(analysis_pca[['individual_id', 'cluster_kmeans', 'cluster_ward'] + feature_cols].head(10))


## Exploratory linear regressions

These are descriptive trend checks. They are not substitutes for the original Coimbra discriminant or logistic equations. The goal is to inspect whether FI/CF, DE/AD, or leading PCs vary with age in the Fishergate sample.


In [ ]:
if 'analysis_pca' not in globals():
    print('Run the PCA cell first.')
else:
    regression_rows = []
    age_reg_df = analysis_pca.dropna(subset=['age_midpoint_years']).copy()
    for var in [c for c in ['FI_CF', 'DE_AD', 'PC1', 'PC2'] if c in age_reg_df.columns]:
        tmp = age_reg_df[['age_midpoint_years', var]].dropna()
        if len(tmp) >= 3:
            slope, intercept = np.polyfit(tmp['age_midpoint_years'], tmp[var], 1)
            regression_rows.append({
                'outcome': var,
                'n': len(tmp),
                'slope_per_year': slope,
                'intercept': intercept,
                'pearson_r': tmp['age_midpoint_years'].corr(tmp[var])
            })
    regression_df = pd.DataFrame(regression_rows)
    regression_df.to_csv(OUT / 'fishergate_age_regression_summaries.csv', index=False)
    display(regression_df)


In [ ]:
if 'analysis_pca' not in globals() or 'cluster_kmeans' not in analysis_pca.columns:
    print('Run PCA and clustering cells first.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, var, color in zip(axes, ['FI_CF', 'DE_AD'], ['darkblue', 'darkgreen']):
        tmp = analysis_pca[['age_midpoint_years', var, 'cluster_kmeans']].dropna()
        ax.scatter(tmp['age_midpoint_years'], tmp[var], c=tmp['cluster_kmeans'], cmap='tab10', s=40, alpha=0.8)
        if len(tmp) >= 3:
            m, b = np.polyfit(tmp['age_midpoint_years'], tmp[var], 1)
            xs = np.linspace(tmp['age_midpoint_years'].min(), tmp['age_midpoint_years'].max(), 100)
            ax.plot(xs, m * xs + b, color=color, lw=2)
        ax.set_title(f'{var} vs age midpoint')
        ax.set_xlabel('Age midpoint (years)')
        ax.set_ylabel(var)
    plt.tight_layout()
    plt.show()


**Independent clustering from ratios alone**

In [ ]:
feature_cols = [c for c in ratio_cols if c in slim_df.columns]
analysis = slim_df[['individual_id'] + [c for c in preferred_meta if c in slim_df.columns and c != 'individual_id'] + feature_cols].copy()
analysis = analysis.dropna(subset=feature_cols).reset_index(drop=True)
print('Rows with complete ratio data:', len(analysis))
X = analysis[feature_cols].to_numpy(dtype=float)
Xz = StandardScaler().fit_transform(X)
pca = PCA()
Xp = pca.fit_transform(Xz)
pc_names = [f'PC{i+1}' for i in range(Xp.shape[1])]
pc_df = pd.DataFrame(Xp, columns=pc_names)
analysis_pca = pd.concat([analysis.reset_index(drop=True), pc_df], axis=1)
analysis_pca.to_csv(OUT / 'fishergate_luna_analysis_pca_scores.csv', index=False)
pd.DataFrame({'component': pc_names, 'explained_variance_ratio': pca.explained_variance_ratio_, 'cumulative_variance_ratio': np.cumsum(pca.explained_variance_ratio_)}).to_csv(OUT / 'fishergate_luna_pca_variance.csv', index=False)
display(analysis_pca.head(5))


In [ ]:
scores = []
for k in range(2, min(7, len(analysis))):
    labels = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(Xz)
    scores.append({'k': k, 'silhouette': silhouette_score(Xz, labels)})
scores_df = pd.DataFrame(scores)
scores_df.to_csv(OUT / 'fishergate_luna_cluster_silhouette.csv', index=False)
display(scores_df)
best_k = int(scores_df.sort_values(['silhouette', 'k'], ascending=[False, True]).iloc[0]['k'])
print('Selected k:', best_k)


In [ ]:
analysis_pca['cluster_kmeans'] = KMeans(n_clusters=best_k, n_init=50, random_state=42).fit_predict(Xz).astype(int)
analysis_pca['cluster_ward'] = AgglomerativeClustering(n_clusters=best_k, linkage='ward').fit_predict(Xz).astype(int)
analysis_pca.to_csv(OUT / 'fishergate_luna_clustered.csv', index=False)
cluster_summary = analysis_pca.groupby('cluster_kmeans')[feature_cols].agg(['mean', 'median', 'count'])
cluster_summary.to_csv(OUT / 'fishergate_luna_cluster_summary.csv')
display(analysis_pca[['individual_id', 'cluster_kmeans', 'cluster_ward'] + feature_cols].head(10))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
var_df = pd.DataFrame({'component': np.arange(1, len(pca.explained_variance_ratio_) + 1), 'explained': pca.explained_variance_ratio_, 'cumulative': np.cumsum(pca.explained_variance_ratio_)})
axes[0].bar(var_df['component'], var_df['explained'], color='slategray')
axes[0].set_title('PCA variance')
axes[0].set_xlabel('Principal component')
axes[0].set_ylabel('Explained variance ratio')
axes[1].scatter(analysis_pca['PC1'], analysis_pca['PC2'], c=analysis_pca['cluster_kmeans'], cmap='tab10', s=50)
for _, r in analysis_pca.iterrows():
    axes[1].annotate(r['individual_id'], (r['PC1'], r['PC2']), fontsize=7, alpha=0.7)
axes[1].set_title('K-means clusters in PCA space')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[2].plot(scores_df['k'], scores_df['silhouette'], marker='o', color='darkred')
axes[2].set_title('Silhouette by k')
axes[2].set_xlabel('k')
axes[2].set_ylabel('Silhouette score')
plt.tight_layout()
plt.show()


In [ ]:
for c in [x for x in ['sex_code', 'age_group', 'preservation', 'source_side', 'measurement_source'] if x in analysis_pca.columns]:
    display(pd.crosstab(analysis_pca['cluster_kmeans'], analysis_pca[c], dropna=False))


## Validation notes
Repeat point placement blinded to prior annotation and biological metadata, with another observer and later session, to assess landmark/ratio reliability. The original paper used two experienced observers, repeated observations at least a week apart, and randomly reordered ilia. This scaffold does not calculate ICC automatically or claim reliable anatomical detection.

FI/CF and DE/AD were the paper's strongest univariate ratios, with section points 0.99 and 0.76. These are not hard biological boundaries and were developed in 34 individuals aged 7–18. No sex decisions are made here. Do not infer male/female from PCA sign or apply logistic output coding without verification.

Geometry audit: A=(gamma_x,delta_y); B=(beta_x,delta_y); C=(alpha_x,delta_y); D=(gamma_x,beta_y); E=beta; F=(alpha_x,beta_y); G=(gamma_x,epsilon_y); H=(beta_x,epsilon_y); I=(alpha_x,epsilon_y). All seven ratios use Euclidean grid-segment lengths. Anatomical point placement, not algebra, is the limiting assumption.

The published Coimbra discriminant function, logistic regression, and section points were applied unchanged to the Fishergate measurements as an external-transfer analysis. Because most Fishergate individuals lacked documented sex and because the sample differs in geographic, chronological, age, and demographic composition from the Coimbra derivation sample, the outputs were treated as provisional sex-linked allocations rather than validated sex determinations. Individuals outside the original 7–18-year derivation range were analyzed separately.